# BERT jailbreak-type classifiers

One fine-tuned `bert-base-uncased` model per evaluated type. The metric dict
keeps Accuracy, F1, AUC, TPR, and FNR. Labels are parsed with the shared
canonical parser, and duplicate prompts are merged before the stratified split.

`RUN_TRAINING=0` skips the fine-tune and still prints the positive counts.
`LABEL_UNLABELLED=0` skips applying the models to the unlabelled DAN prompts.

In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "code").is_dir() else HERE.parent
sys.path.insert(0, str(ROOT / "code"))

import torch
import pandas as pd
from torch.utils.data import Dataset
from transformers import BertForSequenceClassification, BertTokenizer, Trainer, TrainingArguments

from category_models import one_vs_rest_frame
from labels import EVALUATED_JAILBREAK_TYPES, parse_labels
from latex_tables import export_results
from metrics import trainer_compute_metrics
from paths import data_path, results_path

os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
RUN_TRAINING = os.environ.get("RUN_TRAINING", "1") == "1"
LABEL_UNLABELLED = os.environ.get("LABEL_UNLABELLED", "1") == "1"

labelled = pd.read_csv(data_path("df_jailbreak_categ.csv"))
frames = {name: one_vs_rest_frame(labelled, name) for name in EVALUATED_JAILBREAK_TYPES}
for name, frame in frames.items():
    print(f"{name}: positives={int(frame['Jailbreak'].sum())} of {len(frame)}")

class PromptDataset(Dataset):
    def __init__(self, prompts, labels, tokenizer, max_length=128):
        self.prompts = list(prompts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, index):
        encoding = self.tokenizer(
            str(self.prompts[index]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {key: value.squeeze() for key, value in encoding.items()}
        item["labels"] = torch.tensor(self.labels[index], dtype=torch.long)
        return item

def make_training_args(**kwargs):
    try:
        return TrainingArguments(**kwargs)
    except TypeError:
        if "eval_strategy" in kwargs:
            kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
            return TrainingArguments(**kwargs)
        raise

def slug(name):
    return name.lower().replace(" ", "_")

rows = []
if RUN_TRAINING:
    from sklearn.model_selection import train_test_split

    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
    model_dirs = {}
    for name, frame in frames.items():
        print(f"\nTraining {name}")
        train_text, val_text, train_y, val_y = train_test_split(
            frame["prompt"].tolist(),
            frame["Jailbreak"].tolist(),
            test_size=0.2,
            random_state=42,
            stratify=frame["Jailbreak"],
        )
        if set(train_text) & set(val_text):
            raise RuntimeError(f"Prompt leakage while splitting {name}")
        output_dir = ROOT / "models" / "jailbreak_type_bert" / slug(name)
        output_dir.mkdir(parents=True, exist_ok=True)
        model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
        trainer = Trainer(
            model=model,
            args=make_training_args(
                output_dir=str(output_dir / "checkpoints"),
                num_train_epochs=4,
                per_device_train_batch_size=8,
                per_device_eval_batch_size=8,
                warmup_steps=10,
                weight_decay=0.01,
                logging_dir=str(output_dir / "logs"),
                logging_steps=10,
                eval_strategy="epoch",
                save_strategy="no",
                report_to=[],
            ),
            train_dataset=PromptDataset(train_text, train_y, tokenizer),
            eval_dataset=PromptDataset(val_text, val_y, tokenizer),
            compute_metrics=trainer_compute_metrics,
        )
        trainer.train()
        trainer.save_model(str(output_dir))
        tokenizer.save_pretrained(str(output_dir))
        evaluation = trainer.evaluate()
        row = {
            "jailbreak_type": name,
            "accuracy": evaluation["eval_accuracy"],
            "f1": evaluation["eval_f1"],
            "auc": evaluation["eval_auc"],
            "tpr": evaluation["eval_tpr"],
            "fnr": evaluation["eval_fnr"],
            "n_positive_total": int(frame["Jailbreak"].sum()),
            "n_total": int(len(frame)),
        }
        rows.append(row)
        model_dirs[name] = output_dir
        print(name, {key: row[key] for key in ("accuracy", "f1", "auc", "tpr", "fnr")})
    paths = export_results("06_category_bert", "category", rows)
    print(f"Wrote {paths['tex']}")

    if LABEL_UNLABELLED:
        fresh = pd.read_csv(data_path("jailbreak_prompts_DAN.csv"))
        pending = fresh[fresh["Labels"].map(lambda value: len(parse_labels(value)) == 0)].copy()
        loaded = {
            name: BertForSequenceClassification.from_pretrained(str(directory))
            for name, directory in model_dirs.items()
        }
        for model in loaded.values():
            model.eval()
        assigned = []
        for prompt in pending["prompt"].tolist():
            names = []
            inputs = tokenizer(str(prompt), return_tensors="pt", truncation=True, padding=True, max_length=128)
            for name, model in loaded.items():
                with torch.no_grad():
                    logits = model(**inputs).logits
                if int(torch.argmax(logits, dim=-1).item()) == 1:
                    names.append(name)
            assigned.append(", ".join(names))
        pending["Labels"] = assigned
        output_path = results_path("06_category_bert_autolabels.csv")
        pending.to_csv(output_path, index=False)
        print(f"Wrote {len(pending)} labels to {output_path}")
else:
    print("RUN_TRAINING=0, so no category model was fit.")